# Lab 07: Improving HOG-Based Medical Image Classification Using a Genetic Algorithm\n**Author:** Muhammad Asad Mashood  \n**Student ID:** FA23-BAI-039  \n**Dataset:** DermaMNIST  \n**Repository:** [ComputerVision-LABS](https://github.com/asad-mashood/ComputerVision-LABS)  \n\n---\n\n## 1. Executive Summary & Research Question\n> **Research Question:** Can Genetic Algorithm-based feature selection improve the classification performance of HOG features while reducing feature dimensionality on medical images?\n\nIn this laboratory research study, we evaluate whether integrating a **Genetic Algorithm (GA)** for feature selection on top of traditional **Histogram of Oriented Gradients (HOG)** descriptors enhances medical diagnostic classification while eliminating uninformative gradient dimensions. We employ the **DermaMNIST** benchmark.\n

In [1]:
# 1. Imports and Reproducibility Setup\nimport os\nimport json\nimport numpy as np\nimport cv2\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import classification_report, confusion_matrix\n\nprint('Executing Lab 07 Pipeline for DermaMNIST...')\nwith open('results/metrics_comparison.json') as f:\n    results = json.load(f)\nprint('Data and experimental outcomes loaded successfully.')\n

## 2. Table 1: Experimental Configuration and HOG Parameters\nAll models were evaluated under identical, fixed splits with identical preprocessing.\n

In [2]:
# Display Table 1 Configuration\nimport pandas as pd\n\ntable1_data = {\n    'Parameter': [\n        'Medical Dataset', 'Image Resolution', 'HOG Orientations', 'Pixels per Cell',\n        'Cells per Block', 'Block Normalization', 'Feature Selection', 'Classifier',\n        'SVM Kernel', 'Hyperparameter Tuning', 'Data Split', 'Random Seed'\n    ],\n    'Baseline: HOG + SVM': [\n        'DermaMNIST', '128 x 128', '9', '8 x 8',\n        '2 x 2', 'L2-Hys', 'None (Full 2,304 features)', 'SVM',\n        'RBF', 'Validation set', 'Identical fixed split (60/20/20)', '99'\n    ],\n    'Proposed: GA-HOG + SVM': [\n        'DermaMNIST', '128 x 128', '9', '8 x 8',\n        '2 x 2', 'L2-Hys', 'Genetic Algorithm (Setting 2 (Diverse Population GA: Pop=60, Mut=0.09))', 'SVM',\n        'RBF', 'Validation set', 'Identical fixed split (60/20/20)', '99'\n    ]\n}\npd.DataFrame(table1_data)\n

## 3. Table 2: Main Performance Comparison (Without GA vs With GA)\nFull evaluation over unseen test samples across all 9 primary and secondary metrics.\n

In [3]:
# Display Table 2 Performance Results\nb = results['baseline_metrics']\np = results['proposed_metrics']\nimp = results['improvements']\n\ntable2_data = {\n    'Metric': [\n        'Accuracy (%)', 'Precision (Macro, %)', 'Recall (Macro, %)', 'Macro-F1 (%)',\n        'Weighted-F1 (%)', 'Balanced Accuracy (%)', 'Specificity (Macro, %)',\n        'Cohen\'s Kappa', 'Matthews Corr Coeff (MCC)', 'Feature Count', 'Inference Latency (ms)'\n    ],\n    'HOG + SVM (Baseline)': [\n        b['accuracy'], b['precision_macro'], b['recall_macro'], b['macro_f1'],\n        b['weighted_f1'], b['balanced_accuracy'], b['specificity_macro'],\n        b['cohen_kappa'], b['matthews_corrcoef'], results['n_total_features'], b['latency_ms']\n    ],\n    'GA-HOG + SVM (Proposed)': [\n        p['accuracy'], p['precision_macro'], p['recall_macro'], p['macro_f1'],\n        p['weighted_f1'], p['balanced_accuracy'], p['specificity_macro'],\n        p['cohen_kappa'], p['matthews_corrcoef'], results['n_selected_features'], p['latency_ms']\n    ],\n    'Improvement': [\n        f'+{imp["accuracy"]}%', f'+{imp["precision_macro"]}%', f'+{imp["recall_macro"]}%', f'+{imp["macro_f1"]}%',\n        f'+{imp["weighted_f1"]}%', f'+{imp["balanced_accuracy"]}%', f'+{imp["specificity_macro"]}%',\n        f'+{imp["cohen_kappa"]}', f'+{imp["matthews_corrcoef"]}', f'-{results["reduction_pct"]}% pruned', f'{imp["latency_speedup_x"]}x faster'\n    ]\n}\npd.DataFrame(table2_data)\n

## 4. Visualizations and Experimental Analysis\n

In [4]:
# Display Convergence and Confusion Matrices\nfrom IPython.display import Image, display\ndisplay(Image('figures/03_ga_fitness_convergence.png'))\ndisplay(Image('figures/04_feature_reduction_distribution.png'))\ndisplay(Image('figures/05_confusion_matrices.png'))\ndisplay(Image('figures/06_metrics_radar_comparison.png'))\n

## 5. Scientific Discussion & Conclusion\n\n### Strengths of the Proposed Method\n1. **Curse of Dimensionality Mitigation:** Standard HOG generates 2,304 features per 128x128 image. The Genetic Algorithm successfully discarded redundant/noisy background blocks, retaining only **460** high-discriminative features (**80.03% reduction**).\n2. **Superior Generalization:** Validation-guided feature selection prevented the RBF SVM from fitting to high-frequency acquisition artifacts, yielding a **+2.49%** boost in test Macro-F1.\n3. **Inference Latency Speedup:** Kernel dot-product computations scaled down substantially, delivering a **4.76x faster** per-sample classification throughput.\n\n### Limitations & Computational Cost\n- **Evolutionary Overhead:** Evaluating 20 generations of candidate chromosomes requires multiple SVM validation passes during training. While training takes longer than a single baseline pass, this cost is strictly an offline training expenditure.\n- **Stochastic Heuristic:** Genetic search requires careful mutation probability tuning to prevent premature convergence into local sub-optimal feature basins.\n